In [1]:
import sys
import pandas as pd

print(sys.executable)

/home/viktoriiakarpenko/.venvs/mlops-homework-1/bin/python


In [2]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

january_path = data_dir / "yellow_tripdata_2023-01.parquet"
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"

# Download the file only if it is not already available locally.
if not january_path.exists():
    urlretrieve(url, january_path)

print("File is ready")

File is ready


In [3]:
import pandas as pd

df_january = pd.read_parquet(january_path)

# Check the original dimensions before adding new columns.
print("Rows:", df_january.shape[0])
print("Columns:", df_january.shape[1])

df_january.head()

Rows: 3066766
Columns: 19


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [4]:
# Calculate trip duration in minutes.
df_january["duration"] = (
    df_january["tpep_dropoff_datetime"]
    - df_january["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Calculate the standard deviation before removing outliers.
duration_std = df_january["duration"].std()

print(f"Duration standard deviation: {duration_std:.2f} minutes")

df_january[
    ["tpep_pickup_datetime", "tpep_dropoff_datetime", "duration"]
].head()

Duration standard deviation: 42.59 minutes


,tpep_pickup_datetime,tpep_dropoff_datetime,duration
0,2023-01-01 00:32:10,2023-01-01 00:40:36,8.433333
1,2023-01-01 00:55:08,2023-01-01 01:01:27,6.316667
2,2023-01-01 00:25:04,2023-01-01 00:37:49,12.750000
3,2023-01-01 00:03:48,2023-01-01 00:13:25,9.616667
4,2023-01-01 00:10:29,2023-01-01 00:21:19,10.833333


In [5]:
# Keep trips with durations between 1 and 60 minutes, inclusive.
duration_mask = (
    (df_january["duration"] >= 1)
    & (df_january["duration"] <= 60)
)

# Create a separate training dataframe, preserving the original data.
df_train = df_january.loc[duration_mask].copy()

remaining_fraction = len(df_train) / len(df_january)

print("Original rows:", len(df_january))
print("Remaining rows:", len(df_train))
print(f"Remaining percentage: {remaining_fraction:.2%}")

Original rows: 3066766
Remaining rows: 3009173
Remaining percentage: 98.12%


In [6]:
from sklearn.feature_extraction import DictVectorizer

# Treat location IDs as categorical features.
categorical_features = ["PULocationID", "DOLocationID"]
df_train[categorical_features] = df_train[categorical_features].astype(str)

# Convert each trip into a dictionary of feature values.
train_dicts = df_train[categorical_features].to_dict(orient="records")

# Learn the feature mapping and create a sparse feature matrix.
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

print("Training rows:", X_train.shape[0])
print("Feature columns:", X_train.shape[1])
print("Example feature names:", dv.get_feature_names_out()[:10])

Training rows: 3009173
Feature columns: 515
Example feature names: ['DOLocationID=1' 'DOLocationID=10' 'DOLocationID=100' 'DOLocationID=101'
 'DOLocationID=102' 'DOLocationID=106' 'DOLocationID=107'
 'DOLocationID=108' 'DOLocationID=109' 'DOLocationID=11']


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# Use trip duration in minutes as the target.
y_train = df_train["duration"].to_numpy()

# Train a linear regression model with default parameters.
model = LinearRegression()
model.fit(X_train, y_train)

# Predict durations for the training data.
y_train_pred = model.predict(X_train)

# Calculate the training error in minutes.
train_rmse = root_mean_squared_error(y_train, y_train_pred)

print(f"Training RMSE: {train_rmse:.2f} minutes")

Training RMSE: 7.65 minutes


In [8]:
february_path = data_dir / "yellow_tripdata_2023-02.parquet"
february_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet"

if not february_path.exists():
    urlretrieve(february_url, february_path)

df_february = pd.read_parquet(february_path)

print("February rows:", len(df_february))

February rows: 2913955


In [9]:
# Calculate trip duration in minutes.
df_february["duration"] = (
    df_february["tpep_dropoff_datetime"]
    - df_february["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Apply the same duration filter as for the training data.
validation_mask = (
    (df_february["duration"] >= 1)
    & (df_february["duration"] <= 60)
)
df_val = df_february.loc[validation_mask].copy()

# Treat location IDs as categorical features.
df_val[categorical_features] = df_val[categorical_features].astype(str)

val_dicts = df_val[categorical_features].to_dict(orient="records")

# Reuse the vectorizer fitted on January data.
X_val = dv.transform(val_dicts)
y_val = df_val["duration"].to_numpy()

print("Validation matrix shape:", X_val.shape)

Validation matrix shape: (2855951, 515)


In [10]:
# Evaluate the existing model without retraining it.
y_val_pred = model.predict(X_val)

val_rmse = root_mean_squared_error(y_val, y_val_pred)

print(f"Training RMSE: {train_rmse:.2f} minutes")
print(f"Validation RMSE: {val_rmse:.2f} minutes")

Training RMSE: 7.65 minutes
Validation RMSE: 7.81 minutes
